# Usage

This notebook demonstrates how to use the `pnwstore` package to query seismic data from pnwstore1. The package provides a convenient interface to access seismic data, including continuous miniSEED waveforms, station metadata, phase arrivals, and catalogs.

In [ ]:
import obspy

## 1. WaveformClient

The `WaveformClient` class allows you to query continuous miniSEED waveforms from pnwstore1. You can specify the network, station, location, channel, and time range for your query. When you initialize the `WaveformClient`, it will check the sqlite index databases and the local mount for the data year (1980 to 2023 by default). A warning message is shown if anything is missing.

In [2]:
from pnwstore import WaveformClient

client = WaveformClient()

PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


### get_waveforms

In [ ]:
starttime = obspy.UTCDateTime("2022-03-01T09:22:27.350000Z")
endtime   = starttime + 150
s = client.get_waveforms(network="UW", station="SHW", channel="HH?",
                         starttime=starttime, endtime=endtime)
s

3 Trace(s) in Stream:
UW.SHW..HHE | 2022-03-01T09:22:27.350000Z - 2022-03-01T09:24:57.350000Z | 100.0 Hz, 15001 samples
UW.SHW..HHN | 2022-03-01T09:22:27.350000Z - 2022-03-01T09:24:57.350000Z | 100.0 Hz, 15001 samples
UW.SHW..HHZ | 2022-03-01T09:22:27.350000Z - 2022-03-01T09:24:57.350000Z | 100.0 Hz, 15001 samples

### get_waveforms_bulk

In [4]:
t1 = obspy.UTCDateTime("2010-02-27T00:30:00.000")
t2 = obspy.UTCDateTime("2010-02-27T20:30:00.000")

bulk = [("UW", "SHW", "*", "EH*", t1, t2),
        ("UW", "WPO", "*", "?H*", t1, t2),
        ("UW", "SEA", "*", "?NE", t1, t2),
        ("UW", "HBO", "*", "*", t1, t2)]

In [5]:
st = client.get_waveforms_bulk(bulk)
st

23 Trace(s) in Stream:

UW.SHW..EHZ | 2010-02-27T00:30:00.001300Z - 2010-02-27T01:55:39.091300Z | 100.0 Hz, 513910 samples
...
(21 other traces)
...
UW.HBO..EHZ | 2010-02-27T18:32:19.149900Z - 2010-02-27T20:29:59.999900Z | 100.0 Hz, 706086 samples

[Use "print(Stream.__str__(extended=True))" to print all Traces]

## 2. EventClient

The `EventClient` class allows you to query seismic event catalogs from pnwstore1. You can specify the time range, magnitude range, and geographic region for your query. The `get_events` method returns a data frame that match your criteria.

In [8]:
from pnwstore import EventClient

client = EventClient(USERNAME, PASSWORD)

### query

In [14]:
events = client.query(minlatitude=45.9, maxlatitude=46.5,
                      minlongitude=-122.5,maxlongitude= -121.9,)
events = events[events['year']>=2007]
events.head()

,source_id,timestamp,event_type,year,month,day,doy,hour,minute,second,microsecond,latitude,longitude,depth,magnitude,magnitude_type,contributor,number_of_pick
25083,uw10710108,1167625989.320000,earthquake,2007,1,1,1,4,33,9,320000,46.193,-122.192,-1.722,1.1,md,uw,8
25084,uw10710113,1170312168.100000,earthquake,2007,2,1,32,6,42,48,100000,46.195,-122.189,-1.738,1.7,md,uw,12
25085,uw10710118,1167654381.090000,earthquake,2007,1,1,1,12,26,21,90000,46.196,-122.186,-1.866,1.9,md,uw,12
25086,uw10710153,1175387861.630000,earthquake,2007,4,1,91,0,37,41,630000,45.945,-122.359,14.502,1.5,md,uw,18
25087,uw10710173,1170337962.760000,earthquake,2007,2,1,32,13,52,42,760000,46.196,-122.185,-1.597,1.9,md,uw,11


## 3. PickClient

Similarly, the `PickClient` class allows you to query phase arrival picks from pnwstore1. You can specify the time range, network, station, and channel for your query. The `get_picks` method returns a data frame of picks that match your criteria.

In [11]:
from pnwstore import PickClient

client = PickClient(USERNAME, PASSWORD)

### query

In [12]:
picks = client.query()
picks.head()

,pick_id,source_id,network,station,location,channel,timestamp,year,month,day,...,minute,second,microsecond,phase,evaluation_mode,onset,polarity,uncertainty,backazimuth,contributor
0,1,uw10088983,UW,CDF,--,EHZ,341944407.650000,1980,11,1,...,33,27,650000,P,manual,NA,ne,0.03,279.5,UW
1,2,uw10088983,UW,JUN,--,EHZ,341944407.100000,1980,11,1,...,33,27,100000,P,manual,NA,un,0.06,196.5,UW
2,3,uw10088983,UW,LVP,--,EHZ,341944409.310000,1980,11,1,...,33,29,310000,P,manual,NA,po,0.06,69.2,UW
3,4,uw10088983,UW,LVP,--,EHZ,341944412.350000,1980,11,1,...,33,32,350000,S,manual,NA,un,0.15,69.2,UW
4,5,uw10088983,UW,SBL,--,EHZ,341944410.060000,1980,11,1,...,33,30,60000,P,manual,NA,ne,0.06,200.5,UW


## 4. StationClient

The `StationClient` class allows you to query basic station metadata from pnwstore1. You can specify the network, station, location, and channel for your query. The `get_stations` method returns a data frame of station metadata that match your criteria.

In [25]:
from pnwstore import StationClient

client = StationClient(USERNAME, PASSWORD)

### query

In [27]:
stations = client.query(network = "U?", station = "*", channel = "HH?")
stations.head()

,channel_id,network,station,location,channel,latitude,longitude,elevation,depth,starttime,endtime,sampling_rate,azimuth
0,32879,UI,MGOO,01,HHE,47.6491,-117.15,640.00,0.00,1611964800.0,19880899200.0,100.0000,90.00
1,32880,UI,MGOO,01,HHN,47.6491,-117.15,640.00,0.00,1611964800.0,19880899200.0,100.0000,None
2,32881,UI,MGOO,01,HHZ,47.6491,-117.15,640.00,0.00,1611964800.0,19880899200.0,100.0000,None
3,32891,UO,ALSE,--,HHE,44.3818,-123.59,95.00,1.20,1555657200.0,1598387100.0,100.0000,90.00
4,32892,UO,ALSE,--,HHE,44.3818,-123.59,95.00,1.20,1598387100.0,19880899200.0,100.0000,90.00
